# Analisis de resultados del benchmark

Cada caso (`playground`, `mlp`, `cnn`) corre su `bench.py`, que entrena la red
una vez, ejecuta cada backend en su **propio subproceso** y vuelca un
`results.csv` con accuracy, tiempos de setup (`keygen_s`, `compile_s`),
latencia por muestra (`latency_s`) y memoria.

`keygen_s` es la generacion de claves; `compile_s` es la compilacion de la red
al circuito FHE (aproximacion polinomial / cuantizacion / colocacion de
bootstrap). Mapeo por backend: SDK -> `build()` / `Sequential.compile()`;
Concrete-ML -> `fhe_circuit.keygen()` / `compile_torch_model()`;
Orion -> `init_scheme()` / `fit()`+`compile()`.

Dos columnas de VRAM:
- `vram_mb`: pico de memoria del proceso segun NVML (la *huella*). Para la SDK
  esto refleja la **reserva del pool RMM (~90% de la GPU)**, no el uso real.
- `vram_alloc_mb`: pico del *working set real* (bytes efectivamente
  asignados). SDK -> contador del pool RMM (`device_pool_used_bytes`);
  PyTorch -> `torch.cuda.memory_allocated`. Concrete-ML y Orion no exponen un
  contador de asignacion, asi que su `vram_alloc_mb` es 0 (no disponible) y se
  debe leer su `vram_mb`.

Este notebook solo **lee** esos CSV y arma las tablas y figuras de la tesis;
no mide nada. Para regenerar datos, corre el `bench.py` del caso correspondiente.

In [ ]:
import glob
import pandas as pd

paths = sorted(glob.glob('*/results.csv'))
frames = [pd.read_csv(p) for p in paths]
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print('casos:', paths)
df

## Figuras por metrica

La latencia se grafica en escala logaritmica porque el plaintext y los
backends cifrados difieren en varios ordenes de magnitud.

In [ ]:
import matplotlib.pyplot as plt

LOG_METRICS = {'keygen_s', 'compile_s', 'latency_s'}
for metric in ['keygen_s', 'compile_s', 'latency_s', 'vram_mb', 'vram_alloc_mb', 'ram_mb', 'accuracy']:
    if df.empty or metric not in df:
        continue
    pivot = df.pivot(index='backend', columns='case', values=metric)
    pivot.plot(kind='bar', logy=metric in LOG_METRICS, figsize=(6, 4), title=metric)
    plt.tight_layout()
    plt.show()